# 08_Transformer_Encoder.ipynb

This notebook:

1. Uploads `training_dataset.csv`
2. Selects ERA5 + MODIS weather features
3. Builds a Transformer Encoder
4. Trains a weather encoder
5. Saves:
   - `weather_transformer.pth`
   - `transformer_features.csv`


In [ ]:
!pip -q install torch pandas scikit-learn

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

from google.colab import files

print("Upload training_dataset.csv")
uploaded = files.upload()
fname = list(uploaded.keys())[0]

df = pd.read_csv(fname)

weather_cols = [
    "era5_temp_mean_c",
    "era5_dewpoint_mean_c",
    "era5_pressure_mean_pa",
    "era5_u_wind_mean",
    "era5_v_wind_mean",
    "era5_precip_sum_mm",
    "era5_runoff_sum_mm",
    "LST_Day_C"
]

weather_cols = [c for c in weather_cols if c in df.columns]

X = df[weather_cols].fillna(df[weather_cols].median())

# Temporary labels from event_id
labels = []
for e in df["event_id"].astype(str):
    if "FLASH" in e.upper():
        labels.append("FLASH")
    elif "HEAT" in e.upper():
        labels.append("HEAT")
    else:
        labels.append("OTHER")

le = LabelEncoder()
y = le.fit_transform(labels)

class WeatherDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x.values, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

dataset = WeatherDataset(X, y)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

class WeatherTransformer(nn.Module):
    def __init__(self, input_dim, d_model=512, nhead=8, layers=2, classes=2):
        super().__init__()
        self.embed = nn.Linear(input_dim, d_model)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc, num_layers=layers)
        self.classifier = nn.Linear(d_model, classes)

    def forward(self, x):
        x = self.embed(x)
        x = x.unsqueeze(1)
        feat = self.transformer(x).squeeze(1)
        out = self.classifier(feat)
        return out, feat

device = "cuda" if torch.cuda.is_available() else "cpu"

model = WeatherTransformer(
    input_dim=len(weather_cols),
    classes=len(le.classes_)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(10):
    model.train()
    correct = 0
    total = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        pred, _ = model(x_batch)

        loss = criterion(pred, y_batch)

        loss.backward()
        optimizer.step()

        correct += (pred.argmax(1) == y_batch).sum().item()
        total += y_batch.size(0)

    print(f"Epoch {epoch+1}/10 Accuracy: {100*correct/total:.2f}%")

torch.save(model.state_dict(), "weather_transformer.pth")

model.eval()

features = []

with torch.no_grad():
    x_all = torch.tensor(X.values, dtype=torch.float32).to(device)
    _, feat = model(x_all)

feat = feat.cpu().numpy()

out = pd.DataFrame(feat)
out.insert(0, "event_id", df["event_id"])

out.to_csv("transformer_features.csv", index=False)

files.download("weather_transformer.pth")
files.download("transformer_features.csv")

print("Completed.")
